# Quantifying Environmental Volatility
## A Big Data Study of Global Climate Indicators using PySpark
**Name:** Bipin Baruwal  
**Student ID:** 190438  
**Dataset:** https://www.kaggle.com/datasets/bhadramohit/climate-change-dataset

---
### Analysis Modules
1. Environment Setup & SparkSession
2. Data Loading & Schema Inspection
3. Data Cleaning & Preprocessing
4. Exploratory Data Analysis (EDA)
5. Correlation Analysis (CO2 vs Sea Level Rise)
6. Sustainability Impact (Renewable Energy vs Temperature)
7. Anomaly / Extreme-Weather Detection
8. Moving Averages (Rainfall & Temperature — 24-year trend)
9. Linear Regression (Predict Avg Temperature)
10. K-Means Clustering (Environmental Risk Categories)
11. Tableau CSV Exports

In [1]:
# ============================================================
# CELL 1 — Environment Setup & SparkSession
# ============================================================
import os, warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import RegressionEvaluator, ClusteringEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.stat import Correlation

spark = (
    SparkSession.builder
    .appName('ClimateChange_BipinBaruwal_190438')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

# Paths
DATASET_PATH = 'climate_change_dataset.csv'
OUTPUT_DIR   = 'tableau_exports'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=' * 60)
print('  PySpark Climate Change Analytics')
print('  Student: Bipin Baruwal  |  ID: 190438')
print(f'  Spark Version: {spark.version}')
print('=' * 60)

26/03/09 13:31:43 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

  PySpark Climate Change Analytics
  Student: Bipin Baruwal  |  ID: 190438
  Spark Version: 4.1.1


26/03/09 13:31:43 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
# ============================================================
# CELL 2 — Data Loading & Schema Inspection
# ============================================================
raw_df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(DATASET_PATH)
)

# Rename columns to remove spaces and special characters
raw_df = (
    raw_df
    .withColumnRenamed('Avg Temperature (\u00b0C)',        'Avg_Temperature_C')
    .withColumnRenamed('CO2 Emissions (Tons/Capita)',  'CO2_Emissions')
    .withColumnRenamed('Sea Level Rise (mm)',          'Sea_Level_Rise_mm')
    .withColumnRenamed('Rainfall (mm)',                'Rainfall_mm')
    .withColumnRenamed('Renewable Energy (%)',         'Renewable_Energy_Pct')
    .withColumnRenamed('Extreme Weather Events',       'Extreme_Weather_Events')
    .withColumnRenamed('Forest Area (%)',              'Forest_Area_Pct')
)

print('Schema:')
raw_df.printSchema()
print(f'Total Records : {raw_df.count():,}')
print(f'Total Columns : {len(raw_df.columns)}')
print('\nSample Data (top 5 rows):')
raw_df.show(5, truncate=False)
print('Summary Statistics:')
raw_df.describe().show(truncate=False)

Schema:
root
 |-- Year: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- Avg_Temperature_C: double (nullable = true)
 |-- CO2_Emissions: double (nullable = true)
 |-- Sea_Level_Rise_mm: double (nullable = true)
 |-- Rainfall_mm: integer (nullable = true)
 |-- Population: integer (nullable = true)
 |-- Renewable_Energy_Pct: double (nullable = true)
 |-- Extreme_Weather_Events: integer (nullable = true)
 |-- Forest_Area_Pct: double (nullable = true)

Total Records : 1,000
Total Columns : 10

Sample Data (top 5 rows):
+----+---------+-----------------+-------------+-----------------+-----------+----------+--------------------+----------------------+---------------+
|Year|Country  |Avg_Temperature_C|CO2_Emissions|Sea_Level_Rise_mm|Rainfall_mm|Population|Renewable_Energy_Pct|Extreme_Weather_Events|Forest_Area_Pct|
+----+---------+-----------------+-------------+-----------------+-----------+----------+--------------------+----------------------+---------------+
|2006|UK

In [8]:
# ============================================================
# CELL 3 — Data Cleaning & Preprocessing
# ============================================================
KEY_COLS = [
    'Year', 'Country', 'Avg_Temperature_C', 'CO2_Emissions',
    'Sea_Level_Rise_mm', 'Rainfall_mm', 'Renewable_Energy_Pct',
    'Extreme_Weather_Events', 'Forest_Area_Pct', 'Population'
]

print('Null counts per column:')
raw_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in raw_df.columns]).show()

df = raw_df.dropna(subset=KEY_COLS)

numeric_cols = [
    'Avg_Temperature_C', 'CO2_Emissions', 'Sea_Level_Rise_mm',
    'Rainfall_mm', 'Renewable_Energy_Pct', 'Extreme_Weather_Events',
    'Forest_Area_Pct', 'Population', 'Year'
]
for c in numeric_cols:
    df = df.withColumn(c, F.col(c).cast(DoubleType()))

# Derived columns
df = (
    df
    .withColumn('High_Renewable', F.when(F.col('Renewable_Energy_Pct') >= 30, 1).otherwise(0))
    .withColumn('Decade', (F.floor(F.col('Year') / 10) * 10).cast(IntegerType()))
)

print(f'Records after cleaning: {df.count():,}')
df.show(5, truncate=False)

Null counts per column:
+----+-------+-----------------+-------------+-----------------+-----------+----------+--------------------+----------------------+---------------+
|Year|Country|Avg_Temperature_C|CO2_Emissions|Sea_Level_Rise_mm|Rainfall_mm|Population|Renewable_Energy_Pct|Extreme_Weather_Events|Forest_Area_Pct|
+----+-------+-----------------+-------------+-----------------+-----------+----------+--------------------+----------------------+---------------+
|   0|      0|                0|            0|                0|          0|         0|                   0|                     0|              0|
+----+-------+-----------------+-------------+-----------------+-----------+----------+--------------------+----------------------+---------------+

Records after cleaning: 1,000
+------+---------+-----------------+-------------+-----------------+-----------+-------------+--------------------+----------------------+---------------+--------------+------+
|Year  |Country  |Avg_Temper

In [9]:
# ============================================================
# CELL 4 — Exploratory Data Analysis (EDA)
# ============================================================
print('Records per Country:')
df.groupBy('Country').count().orderBy(F.desc('count')).show(20)

print('Year-wise Global Average Temperature (\u00b0C):')
df.groupBy('Year').agg(
    F.round(F.avg('Avg_Temperature_C'), 2).alias('Global_Avg_Temp')
).orderBy('Year').show(25)

print('Country-wise Average CO2 Emissions (Tons/Capita):')
df.groupBy('Country').agg(
    F.round(F.avg('CO2_Emissions'), 2).alias('Avg_CO2')
).orderBy(F.desc('Avg_CO2')).show()

print('Country-wise Average Renewable Energy %:')
df.groupBy('Country').agg(
    F.round(F.avg('Renewable_Energy_Pct'), 2).alias('Avg_Renewable_Pct')
).orderBy(F.desc('Avg_Renewable_Pct')).show()

print('Decade-wise Climate Summary:')
df.groupBy('Decade').agg(
    F.round(F.avg('Avg_Temperature_C'),   2).alias('Avg_Temp'),
    F.round(F.avg('CO2_Emissions'),        2).alias('Avg_CO2'),
    F.round(F.avg('Sea_Level_Rise_mm'),    2).alias('Avg_SeaLevel'),
    F.round(F.avg('Rainfall_mm'),          2).alias('Avg_Rainfall'),
    F.round(F.avg('Renewable_Energy_Pct'), 2).alias('Avg_Renewable_Pct'),
    F.count('*').alias('Record_Count'),
).orderBy('Decade').show()

Records per Country:
+------------+-----+
|     Country|count|
+------------+-----+
|   Indonesia|   75|
|      Russia|   74|
|South Africa|   73|
|         USA|   73|
|       India|   70|
|       China|   67|
|   Argentina|   67|
|      Brazil|   67|
|      Canada|   67|
|      France|   66|
|          UK|   65|
|       Japan|   63|
|     Germany|   61|
|   Australia|   57|
|      Mexico|   55|
+------------+-----+

Year-wise Global Average Temperature (°C):
+------+---------------+
|  Year|Global_Avg_Temp|
+------+---------------+
|2000.0|           20.5|
|2001.0|          20.12|
|2002.0|          21.43|
|2003.0|          18.22|
|2004.0|           18.8|
|2005.0|          19.53|
|2006.0|          19.81|
|2007.0|          20.56|
|2008.0|          19.14|
|2009.0|          19.73|
|2010.0|          17.96|
|2011.0|          18.86|
|2012.0|          17.57|
|2013.0|          18.83|
|2014.0|          20.83|
|2015.0|          19.92|
|2016.0|          21.16|
|2017.0|          19.48|
|2018.0|   

In [10]:
# ============================================================
# CELL 5 — Correlation Analysis (CO2 vs Sea Level Rise)
# ============================================================

# Global Pearson correlation
co2_sea_corr   = df.stat.corr('CO2_Emissions',        'Sea_Level_Rise_mm', 'pearson')
temp_co2_corr  = df.stat.corr('CO2_Emissions',        'Avg_Temperature_C', 'pearson')
forest_temp_corr = df.stat.corr('Forest_Area_Pct',    'Avg_Temperature_C', 'pearson')
rain_temp_corr = df.stat.corr('Rainfall_mm',          'Avg_Temperature_C', 'pearson')
renew_co2_corr = df.stat.corr('Renewable_Energy_Pct', 'CO2_Emissions',     'pearson')

print('=' * 55)
print('  Global Correlation Results (Pearson)')
print('=' * 55)
print(f'  CO2 Emissions    vs Sea Level Rise  : {co2_sea_corr:+.4f}')
print(f'  CO2 Emissions    vs Avg Temperature : {temp_co2_corr:+.4f}')
print(f'  Forest Area %    vs Avg Temperature : {forest_temp_corr:+.4f}')
print(f'  Rainfall (mm)    vs Avg Temperature : {rain_temp_corr:+.4f}')
print(f'  Renewable Energy vs CO2 Emissions   : {renew_co2_corr:+.4f}')

# Country-level CO2 vs Sea Level correlation
print('\nCountry-level Pearson Correlation (CO2 vs Sea Level Rise):')
countries = [r['Country'] for r in df.select('Country').distinct().collect()]
country_corrs = []
for country in sorted(countries):
    sub = df.filter(F.col('Country') == country)
    if sub.count() > 1:
        c = sub.stat.corr('CO2_Emissions', 'Sea_Level_Rise_mm', 'pearson')
        country_corrs.append((country, round(c, 4)))
spark.createDataFrame(country_corrs, ['Country', 'CO2_SeaLevel_Corr']) \
     .orderBy(F.desc('CO2_SeaLevel_Corr')).show(truncate=False)

# MLlib multi-feature correlation matrix
print('Multi-Feature Pearson Correlation Matrix:')
corr_features = [
    'CO2_Emissions', 'Sea_Level_Rise_mm', 'Avg_Temperature_C',
    'Rainfall_mm', 'Renewable_Energy_Pct', 'Forest_Area_Pct', 'Extreme_Weather_Events'
]
vec_df = VectorAssembler(inputCols=corr_features, outputCol='corr_vec').transform(df)
corr_matrix = Correlation.corr(vec_df, 'corr_vec', 'pearson')
print('Features:', corr_features)
print(corr_matrix.head()[0])

  Global Correlation Results (Pearson)
  CO2 Emissions    vs Sea Level Rise  : -0.0388
  CO2 Emissions    vs Avg Temperature : +0.0123
  Forest Area %    vs Avg Temperature : -0.0170
  Rainfall (mm)    vs Avg Temperature : -0.0045
  Renewable Energy vs CO2 Emissions   : -0.0234

Country-level Pearson Correlation (CO2 vs Sea Level Rise):
+------------+-----------------+
|Country     |CO2_SeaLevel_Corr|
+------------+-----------------+
|Mexico      |0.1165           |
|Russia      |0.0874           |
|Indonesia   |0.0785           |
|India       |0.0381           |
|South Africa|0.0323           |
|USA         |0.0272           |
|Germany     |-0.0349          |
|Australia   |-0.0754          |
|Brazil      |-0.0851          |
|Argentina   |-0.089           |
|Canada      |-0.0893          |
|Japan       |-0.1052          |
|China       |-0.1285          |
|UK          |-0.1753          |
|France      |-0.1759          |
+------------+-----------------+

Multi-Feature Pearson Correlation

In [11]:
# ============================================================
# CELL 6 — Sustainability Impact: Renewable Energy vs Temperature
# ============================================================
print('High Renewable (>=30%) vs Low Renewable (<30%) Climate Comparison:')
df.groupBy('High_Renewable').agg(
    F.round(F.avg('Avg_Temperature_C'),      2).alias('Avg_Temp'),
    F.round(F.avg('CO2_Emissions'),           2).alias('Avg_CO2'),
    F.round(F.avg('Sea_Level_Rise_mm'),       2).alias('Avg_SeaLevel'),
    F.round(F.avg('Extreme_Weather_Events'),  2).alias('Avg_ExtremeEvents'),
    F.count('*').alias('Record_Count'),
).orderBy('High_Renewable').show(truncate=False)

print('Country-level: Avg Renewable Energy % vs Avg Temperature:')
df.groupBy('Country').agg(
    F.round(F.avg('Renewable_Energy_Pct'), 2).alias('Avg_Renewable'),
    F.round(F.avg('Avg_Temperature_C'),    2).alias('Avg_Temp'),
    F.round(F.avg('CO2_Emissions'),        2).alias('Avg_CO2'),
).orderBy(F.desc('Avg_Renewable')).show(truncate=False)

# Quartile analysis
q1, q2, q3 = df.approxQuantile('Renewable_Energy_Pct', [0.25, 0.5, 0.75], 0.01)
print(f'Renewable Energy Quartiles — Q1={q1:.1f}%  Q2={q2:.1f}%  Q3={q3:.1f}%')

df_q = df.withColumn(
    'Renewable_Quartile',
    F.when(F.col('Renewable_Energy_Pct') <= q1, 'Q1_Low')
     .when(F.col('Renewable_Energy_Pct') <= q2, 'Q2_MedLow')
     .when(F.col('Renewable_Energy_Pct') <= q3, 'Q3_MedHigh')
     .otherwise('Q4_High')
)
print('Temperature by Renewable Energy Quartile:')
df_q.groupBy('Renewable_Quartile').agg(
    F.round(F.avg('Avg_Temperature_C'), 2).alias('Avg_Temp'),
    F.round(F.avg('CO2_Emissions'),      2).alias('Avg_CO2'),
    F.count('*').alias('Count'),
).orderBy('Renewable_Quartile').show()

High Renewable (>=30%) vs Low Renewable (<30%) Climate Comparison:
+--------------+--------+-------+------------+-----------------+------------+
|High_Renewable|Avg_Temp|Avg_CO2|Avg_SeaLevel|Avg_ExtremeEvents|Record_Count|
+--------------+--------+-------+------------+-----------------+------------+
|0             |20.4    |10.63  |3.01        |7.32             |574         |
|1             |19.18   |10.15  |3.01        |7.25             |426         |
+--------------+--------+-------+------------+-----------------+------------+

Country-level: Avg Renewable Energy % vs Avg Temperature:
+------------+-------------+--------+-------+
|Country     |Avg_Renewable|Avg_Temp|Avg_CO2|
+------------+-------------+--------+-------+
|China       |29.13        |20.28   |10.84  |
|Brazil      |28.79        |20.84   |10.84  |
|France      |28.74        |19.38   |10.97  |
|Germany     |28.71        |20.29   |9.88   |
|Russia      |28.51        |20.72   |8.97   |
|UK          |27.92        |18.49   |1

In [12]:
# ============================================================
# CELL 7 — Anomaly & Extreme-Weather Detection
# ============================================================

# Compute global mean and stddev for each metric
stats = df.select(
    F.mean('Avg_Temperature_C').alias('mu_temp'), F.stddev('Avg_Temperature_C').alias('sd_temp'),
    F.mean('CO2_Emissions').alias('mu_co2'),      F.stddev('CO2_Emissions').alias('sd_co2'),
    F.mean('Sea_Level_Rise_mm').alias('mu_sea'),  F.stddev('Sea_Level_Rise_mm').alias('sd_sea'),
    F.mean('Rainfall_mm').alias('mu_rain'),       F.stddev('Rainfall_mm').alias('sd_rain'),
).collect()[0]

print('Thresholds (Mean +/- 2*StdDev):')
print(f'  Avg Temperature  : mu={stats.mu_temp:.2f}  sd={stats.sd_temp:.2f}  => outlier outside [{stats.mu_temp-2*stats.sd_temp:.2f}, {stats.mu_temp+2*stats.sd_temp:.2f}]')
print(f'  CO2 Emissions    : mu={stats.mu_co2:.2f}   sd={stats.sd_co2:.2f}  => outlier > {stats.mu_co2+2*stats.sd_co2:.2f}')
print(f'  Sea Level Rise   : mu={stats.mu_sea:.2f}   sd={stats.sd_sea:.2f}  => outlier outside [{stats.mu_sea-2*stats.sd_sea:.2f}, {stats.mu_sea+2*stats.sd_sea:.2f}]')
print(f'  Rainfall (mm)    : mu={stats.mu_rain:.2f}  sd={stats.sd_rain:.2f} => outlier outside [{stats.mu_rain-2*stats.sd_rain:.2f}, {stats.mu_rain+2*stats.sd_rain:.2f}]')

# Flag anomalies
df_anomaly = (
    df
    .withColumn('Temp_Anomaly',
        F.when((F.col('Avg_Temperature_C') > stats.mu_temp + 2*stats.sd_temp) |
               (F.col('Avg_Temperature_C') < stats.mu_temp - 2*stats.sd_temp), 1).otherwise(0))
    .withColumn('CO2_Anomaly',
        F.when(F.col('CO2_Emissions') > stats.mu_co2 + 2*stats.sd_co2, 1).otherwise(0))
    .withColumn('SeaLevel_Anomaly',
        F.when((F.col('Sea_Level_Rise_mm') > stats.mu_sea + 2*stats.sd_sea) |
               (F.col('Sea_Level_Rise_mm') < stats.mu_sea - 2*stats.sd_sea), 1).otherwise(0))
    .withColumn('Rainfall_Anomaly',
        F.when((F.col('Rainfall_mm') > stats.mu_rain + 2*stats.sd_rain) |
               (F.col('Rainfall_mm') < stats.mu_rain - 2*stats.sd_rain), 1).otherwise(0))
    .withColumn('Anomaly_Score',
        F.col('Temp_Anomaly') + F.col('CO2_Anomaly') +
        F.col('SeaLevel_Anomaly') + F.col('Rainfall_Anomaly'))
    .withColumn('Extreme_Record', F.when(F.col('Anomaly_Score') >= 1, 1).otherwise(0))
)

total  = df_anomaly.count()
extreme= df_anomaly.filter(F.col('Extreme_Record') == 1).count()
print(f'\nExtreme Records (>= 1 anomaly flag): {extreme:,} / {total:,}  ({100*extreme/total:.1f}%)')

print('\nTop 20 Most Extreme Records:')
df_anomaly.filter(F.col('Extreme_Record') == 1) \
    .select('Year','Country','Avg_Temperature_C','CO2_Emissions','Sea_Level_Rise_mm',
            'Rainfall_mm','Temp_Anomaly','CO2_Anomaly','SeaLevel_Anomaly','Rainfall_Anomaly','Anomaly_Score') \
    .orderBy(F.desc('Anomaly_Score')).show(20, truncate=False)

print('Country-level Extreme Record Summary:')
df_anomaly.groupBy('Country').agg(
    F.sum('Extreme_Record').alias('Extreme_Records'),
    F.count('*').alias('Total_Records'),
    F.round(F.sum('Extreme_Record') / F.count('*') * 100, 1).alias('Pct_Extreme'),
).orderBy(F.desc('Extreme_Records')).show(truncate=False)

Thresholds (Mean +/- 2*StdDev):
  Avg Temperature  : mu=19.88  sd=8.54  => outlier outside [2.80, 36.97]
  CO2 Emissions    : mu=10.43   sd=5.61  => outlier > 21.66
  Sea Level Rise   : mu=3.01   sd=1.15  => outlier outside [0.72, 5.30]
  Rainfall (mm)    : mu=1738.76  sd=708.98 => outlier outside [320.81, 3156.71]

Extreme Records (>= 1 anomaly flag): 0 / 1,000  (0.0%)

Top 20 Most Extreme Records:
+----+-------+-----------------+-------------+-----------------+-----------+------------+-----------+----------------+----------------+-------------+
|Year|Country|Avg_Temperature_C|CO2_Emissions|Sea_Level_Rise_mm|Rainfall_mm|Temp_Anomaly|CO2_Anomaly|SeaLevel_Anomaly|Rainfall_Anomaly|Anomaly_Score|
+----+-------+-----------------+-------------+-----------------+-----------+------------+-----------+----------------+----------------+-------------+
+----+-------+-----------------+-------------+-----------------+-----------+------------+-----------+----------------+----------------+------------

In [13]:
# ============================================================
# CELL 8 — Moving Averages (Rainfall & Temperature, 24 years)
# ============================================================

# Year-level global aggregation
yearly = (
    df.groupBy('Year')
      .agg(
          F.round(F.avg('Avg_Temperature_C'), 3).alias('Yearly_Avg_Temp'),
          F.round(F.avg('Rainfall_mm'),        3).alias('Yearly_Avg_Rainfall'),
          F.round(F.avg('CO2_Emissions'),       3).alias('Yearly_Avg_CO2'),
          F.round(F.avg('Sea_Level_Rise_mm'),   3).alias('Yearly_Avg_SeaLevel'),
      )
      .orderBy('Year')
)

# Window specs
w3 = Window.orderBy('Year').rowsBetween(-1, 1)   # 3-year centred
w5 = Window.orderBy('Year').rowsBetween(-2, 2)   # 5-year centred

yearly_ma = (
    yearly
    .withColumn('MA3_Temp',      F.round(F.avg('Yearly_Avg_Temp').over(w3),     3))
    .withColumn('MA5_Temp',      F.round(F.avg('Yearly_Avg_Temp').over(w5),     3))
    .withColumn('MA3_Rainfall',  F.round(F.avg('Yearly_Avg_Rainfall').over(w3), 3))
    .withColumn('MA5_Rainfall',  F.round(F.avg('Yearly_Avg_Rainfall').over(w5), 3))
    .withColumn('MA3_CO2',       F.round(F.avg('Yearly_Avg_CO2').over(w3),      3))
    .withColumn('MA5_CO2',       F.round(F.avg('Yearly_Avg_CO2').over(w5),      3))
    .withColumn('MA3_SeaLevel',  F.round(F.avg('Yearly_Avg_SeaLevel').over(w3), 3))
    .withColumn('MA5_SeaLevel',  F.round(F.avg('Yearly_Avg_SeaLevel').over(w5), 3))
)

print('Global Year-wise Moving Averages (3-year & 5-year):')
yearly_ma.show(30, truncate=False)

# Country-level 3-year moving average
wc3 = Window.partitionBy('Country').orderBy('Year').rowsBetween(-1, 1)
country_ma = (
    df.withColumn('MA3_Country_Temp',     F.round(F.avg('Avg_Temperature_C').over(wc3), 3))
      .withColumn('MA3_Country_Rainfall', F.round(F.avg('Rainfall_mm').over(wc3),        3))
      .select('Year','Country','Avg_Temperature_C','MA3_Country_Temp','Rainfall_mm','MA3_Country_Rainfall')
      .orderBy('Country','Year')
)
print('Country-level 3-Year Moving Averages:')
country_ma.show(30, truncate=False)

# First vs Last 5-year period
print('Long-term Trend — First Period (2000-2004) vs Last Period (2019-2023):')
for label, filt in [('2000-2004', F.col('Year') <= 2004), ('2019-2023', F.col('Year') >= 2019)]:
    print(f'  Period: {label}')
    df.filter(filt).agg(
        F.round(F.avg('Avg_Temperature_C'), 2).alias('Avg_Temp'),
        F.round(F.avg('Rainfall_mm'),        2).alias('Avg_Rainfall'),
        F.round(F.avg('CO2_Emissions'),      2).alias('Avg_CO2'),
        F.round(F.avg('Sea_Level_Rise_mm'),  2).alias('Avg_SeaLevel'),
    ).show()

Global Year-wise Moving Averages (3-year & 5-year):
+------+---------------+-------------------+--------------+-------------------+--------+--------+------------+------------+-------+-------+------------+------------+
|Year  |Yearly_Avg_Temp|Yearly_Avg_Rainfall|Yearly_Avg_CO2|Yearly_Avg_SeaLevel|MA3_Temp|MA5_Temp|MA3_Rainfall|MA5_Rainfall|MA3_CO2|MA5_CO2|MA3_SeaLevel|MA5_SeaLevel|
+------+---------------+-------------------+--------------+-------------------+--------+--------+------------+------------+-------+-------+------------+------------+
|2000.0|20.502         |1687.736           |11.225        |2.94               |20.31   |20.684  |1722.319    |1791.128    |10.577 |10.362 |3.107       |3.055       |
|2001.0|20.117         |1756.902           |9.929         |3.273              |20.684  |20.068  |1791.128    |1800.023    |10.362 |10.543 |3.055       |3.025       |
|2002.0|21.433         |1928.745           |9.933         |2.953              |19.923  |19.815  |1837.451    |1782.802

In [14]:
# ============================================================
# CELL 9 — Linear Regression (Predict Avg Temperature)
# ============================================================
LR_FEATURES = [
    'CO2_Emissions', 'Forest_Area_Pct', 'Renewable_Energy_Pct',
    'Rainfall_mm', 'Sea_Level_Rise_mm', 'Extreme_Weather_Events', 'Year'
]
LR_TARGET = 'Avg_Temperature_C'

lr_assembler = VectorAssembler(inputCols=LR_FEATURES, outputCol='raw_feats', handleInvalid='skip')
lr_scaler    = StandardScaler(inputCol='raw_feats', outputCol='features', withMean=True, withStd=True)
lr_model     = LinearRegression(featuresCol='features', labelCol=LR_TARGET,
                                 maxIter=100, regParam=0.1, elasticNetParam=0.0)

lr_pipeline = Pipeline(stages=[lr_assembler, lr_scaler, lr_model])

lr_df              = df.select(LR_FEATURES + [LR_TARGET]).dropna()
train_lr, test_lr  = lr_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_lr.count():,}  |  Test: {test_lr.count():,}')

lr_fitted      = lr_pipeline.fit(train_lr)
lr_predictions = lr_fitted.transform(test_lr)

rmse = RegressionEvaluator(labelCol=LR_TARGET, predictionCol='prediction', metricName='rmse').evaluate(lr_predictions)
mae  = RegressionEvaluator(labelCol=LR_TARGET, predictionCol='prediction', metricName='mae').evaluate(lr_predictions)
r2   = RegressionEvaluator(labelCol=LR_TARGET, predictionCol='prediction', metricName='r2').evaluate(lr_predictions)

print('\n' + '=' * 45)
print('  Linear Regression Results (Test Set)')
print('=' * 45)
print(f'  RMSE : {rmse:.4f} \u00b0C')
print(f'  MAE  : {mae:.4f} \u00b0C')
print(f'  R\u00b2   : {r2:.4f}')

inner_lr = lr_fitted.stages[-1]
print(f'\n  Intercept : {inner_lr.intercept:.4f}')
print('  Coefficients (standardised):')
for feat, coeff in zip(LR_FEATURES, inner_lr.coefficients):
    print(f'    {feat:<30}: {coeff:+.4f}')

print('\nSample Predictions vs Actuals:')
lr_predictions.select(
    LR_TARGET, F.round('prediction', 3).alias('Predicted'),
    F.round(F.abs(F.col(LR_TARGET) - F.col('prediction')), 3).alias('AbsError')
).show(15)

Train: 838  |  Test: 162

  Linear Regression Results (Test Set)
  RMSE : 8.5251 °C
  MAE  : 7.3221 °C
  R²   : -0.0186

  Intercept : 19.9130
  Coefficients (standardised):
    CO2_Emissions                 : +0.0989
    Forest_Area_Pct               : -0.2148
    Renewable_Energy_Pct          : -0.4682
    Rainfall_mm                   : -0.2132
    Sea_Level_Rise_mm             : +0.5514
    Extreme_Weather_Events        : +0.5336
    Year                          : +0.1114

Sample Predictions vs Actuals:
+-----------------+---------+--------+
|Avg_Temperature_C|Predicted|AbsError|
+-----------------+---------+--------+
|              9.8|   19.056|   9.256|
|             12.4|   20.427|   8.027|
|             15.3|   18.286|   2.986|
|             26.6|   20.132|   6.468|
|             28.5|    21.01|    7.49|
|              6.4|   19.329|  12.929|
|             21.5|   19.839|   1.661|
|             19.9|   19.426|   0.474|
|             29.6|   20.334|   9.266|
|             17.3

In [15]:
# ============================================================
# CELL 10 — K-Means Clustering (Environmental Risk Categories)
# ============================================================
CLUSTER_FEATURES = [
    'CO2_Emissions', 'Sea_Level_Rise_mm', 'Avg_Temperature_C',
    'Renewable_Energy_Pct', 'Extreme_Weather_Events', 'Forest_Area_Pct'
]

km_assembler = VectorAssembler(inputCols=CLUSTER_FEATURES, outputCol='km_raw', handleInvalid='skip')
km_scaler    = StandardScaler(inputCol='km_raw', outputCol='km_features', withMean=True, withStd=True)

km_clean = df.dropna(subset=CLUSTER_FEATURES)
km_prep  = km_assembler.transform(km_clean)
km_prep  = km_scaler.fit(km_prep).transform(km_prep)

# Elbow Method
print('Elbow Method — WSSSE for K=2..7:')
for k in range(2, 8):
    km_test = KMeans(featuresCol='km_features', k=k, seed=42, maxIter=20)
    km_fit  = km_test.fit(km_prep)
    print(f'  K={k}  WSSSE={km_fit.summary.trainingCost:.2f}')

# Final model with K=4
K = 4
km_pipeline  = Pipeline(stages=[
    km_assembler,
    km_scaler,
    KMeans(featuresCol='km_features', k=K, seed=42, maxIter=50)
])
km_fitted    = km_pipeline.fit(km_clean)
km_preds     = km_fitted.transform(km_clean).withColumnRenamed('prediction', 'Cluster')

silhouette = ClusteringEvaluator(featuresCol='km_features', predictionCol='Cluster').evaluate(km_preds)
print(f'\nK-Means (K={K}) Silhouette Score: {silhouette:.4f}')

print(f'\nCluster Profiles (K={K}):')
km_preds.groupBy('Cluster').agg(
    F.count('*').alias('Count'),
    F.round(F.avg('CO2_Emissions'),           2).alias('Avg_CO2'),
    F.round(F.avg('Sea_Level_Rise_mm'),        2).alias('Avg_SeaLevel'),
    F.round(F.avg('Avg_Temperature_C'),        2).alias('Avg_Temp'),
    F.round(F.avg('Renewable_Energy_Pct'),     2).alias('Avg_Renewable'),
    F.round(F.avg('Extreme_Weather_Events'),   2).alias('Avg_ExtremeEvents'),
    F.round(F.avg('Forest_Area_Pct'),          2).alias('Avg_Forest'),
).orderBy('Cluster').show(truncate=False)

# Assign risk labels based on CO2
cluster_summary = (
    km_preds.groupBy('Cluster').agg(F.avg('CO2_Emissions').alias('Avg_CO2'))
            .orderBy(F.desc('Avg_CO2')).collect()
)
risk_labels = {cluster_summary[i]['Cluster']: lbl for i, lbl in
               enumerate(['High Risk', 'Medium-High Risk', 'Medium-Low Risk', 'Low Risk'])}
print(f'Risk Label Mapping (by CO2): {risk_labels}')

label_map = F.create_map(*[item for pair in [(F.lit(k), F.lit(v)) for k,v in risk_labels.items()] for item in pair])
km_preds  = km_preds.withColumn('Risk_Category', label_map[F.col('Cluster')])

print('\nRisk Category Distribution by Country:')
km_preds.groupBy('Country', 'Risk_Category').count().orderBy('Country').show(30, truncate=False)

Elbow Method — WSSSE for K=2..7:
  K=2  WSSSE=5260.92
  K=3  WSSSE=4741.24
  K=4  WSSSE=4333.16
  K=5  WSSSE=4022.75
  K=6  WSSSE=3781.91
  K=7  WSSSE=3555.79

K-Means (K=4) Silhouette Score: 0.2165

Cluster Profiles (K=4):
+-------+-----+-------+------------+--------+-------------+-----------------+----------+
|Cluster|Count|Avg_CO2|Avg_SeaLevel|Avg_Temp|Avg_Renewable|Avg_ExtremeEvents|Avg_Forest|
+-------+-----+-------+------------+--------+-------------+-----------------+----------+
|0      |239  |14.76  |3.36        |27.22   |29.51        |9.05             |36.37     |
|1      |269  |14.77  |2.56        |12.75   |25.31        |6.41             |45.06     |
|2      |242  |6.15   |3.33        |23.35   |15.95        |6.22             |46.22     |
|3      |250  |5.75   |2.85        |17.19   |38.32        |7.6              |34.29     |
+-------+-----+-------+------------+--------+-------------+-----------------+----------+

Risk Label Mapping (by CO2): {1: 'High Risk', 0: 'Medium-High R

In [16]:
# ============================================================
# CELL 11 — Tableau CSV Exports
# ============================================================
import glob, shutil

def export_csv(spark_df, filename, description):
    """Write PySpark DataFrame as a single CSV for Tableau."""
    tmp_path  = os.path.join(OUTPUT_DIR, filename + '_tmp')
    final_path = os.path.join(OUTPUT_DIR, filename)
    spark_df.coalesce(1).write.mode('overwrite').option('header','true').csv(tmp_path)
    parts = glob.glob(os.path.join(tmp_path, 'part-*.csv'))
    if parts:
        shutil.copy(parts[0], final_path)
        shutil.rmtree(tmp_path, ignore_errors=True)
    print(f'  Exported: {filename}  ({description})')

# T1 — Global Environmental Map
export_csv(
    df.select('Year','Country','CO2_Emissions','Avg_Temperature_C',
              'Sea_Level_Rise_mm','Rainfall_mm','Population',
              'Renewable_Energy_Pct','Extreme_Weather_Events','Forest_Area_Pct'),
    'T1_global_environmental_map.csv',
    'Global Map — CO2 Emissions per Country'
)

# T2 — Sea Level vs Population Trend
export_csv(
    df.groupBy('Year').agg(
        F.round(F.avg('Sea_Level_Rise_mm'), 3).alias('Avg_Sea_Level_Rise_mm'),
        F.sum('Population').alias('Total_Population'),
        F.round(F.avg('Avg_Temperature_C'), 3).alias('Avg_Temperature_C'),
        F.round(F.avg('CO2_Emissions'), 3).alias('Avg_CO2'),
    ).orderBy('Year'),
    'T2_sea_level_population_trend.csv',
    'Trend Dashboard — Sea Level vs Population Growth'
)

# T3 — Energy Pivot by Country
export_csv(
    df.groupBy('Country').agg(
        F.round(F.avg('Renewable_Energy_Pct'), 2).alias('Avg_Renewable_Pct'),
        F.round(100 - F.avg('Renewable_Energy_Pct'), 2).alias('Avg_NonRenewable_Pct'),
        F.round(F.avg('CO2_Emissions'), 2).alias('Avg_CO2'),
        F.round(F.avg('Avg_Temperature_C'), 2).alias('Avg_Temp'),
    ).orderBy(F.desc('Avg_Renewable_Pct')),
    'T3_energy_pivot_by_country.csv',
    'Energy Pivot — Renewable vs Non-Renewable by Region'
)

# T4 — Moving Averages
export_csv(yearly_ma, 'T4_moving_averages_trend.csv', '3-year & 5-year Moving Averages')

# T5 — Anomaly Records
export_csv(
    df_anomaly.select('Year','Country','Avg_Temperature_C','CO2_Emissions',
                      'Sea_Level_Rise_mm','Rainfall_mm','Extreme_Weather_Events',
                      'Temp_Anomaly','CO2_Anomaly','SeaLevel_Anomaly',
                      'Rainfall_Anomaly','Anomaly_Score','Extreme_Record'),
    'T5_anomaly_extreme_weather.csv',
    'Anomaly Detection — Extreme Weather Records'
)

# T6 — K-Means Risk Categories
export_csv(
    km_preds.select('Year','Country','CO2_Emissions','Sea_Level_Rise_mm',
                    'Avg_Temperature_C','Renewable_Energy_Pct',
                    'Extreme_Weather_Events','Forest_Area_Pct','Cluster','Risk_Category'),
    'T6_kmeans_risk_categories.csv',
    'K-Means Risk Category Assignments'
)

# T7 — Regression Predictions
export_csv(
    lr_predictions.select(
        LR_TARGET,
        F.round('prediction', 3).alias('Predicted_Temp'),
        F.round(F.abs(F.col(LR_TARGET) - F.col('prediction')), 3).alias('AbsError'),
        *LR_FEATURES
    ),
    'T7_regression_predictions.csv',
    'Linear Regression Predictions vs Actuals'
)

print(f'\nAll Tableau CSVs saved to: ./{OUTPUT_DIR}/')

  Exported: T1_global_environmental_map.csv  (Global Map — CO2 Emissions per Country)
  Exported: T2_sea_level_population_trend.csv  (Trend Dashboard — Sea Level vs Population Growth)
  Exported: T3_energy_pivot_by_country.csv  (Energy Pivot — Renewable vs Non-Renewable by Region)
  Exported: T4_moving_averages_trend.csv  (3-year & 5-year Moving Averages)
  Exported: T5_anomaly_extreme_weather.csv  (Anomaly Detection — Extreme Weather Records)
  Exported: T6_kmeans_risk_categories.csv  (K-Means Risk Category Assignments)
  Exported: T7_regression_predictions.csv  (Linear Regression Predictions vs Actuals)

All Tableau CSVs saved to: ./tableau_exports/


In [17]:
# ============================================================
# CELL 12 — Final Summary
# ============================================================
print('=' * 60)
print('  ANALYSIS COMPLETE — KEY FINDINGS SUMMARY')
print('  Student: Bipin Baruwal  |  ID: 190438')
print('=' * 60)

year_min = int(df.agg(F.min('Year')).collect()[0][0])
year_max = int(df.agg(F.max('Year')).collect()[0][0])
n_countries = df.select('Country').distinct().count()

print(f'''
  Dataset
  -------
  Records    : {df.count():,}
  Countries  : {n_countries}
  Year Range : {year_min} - {year_max}  ({year_max - year_min + 1} years)

  [Cell 5] Correlation Analysis
  CO2 Emissions    vs Sea Level Rise  : {co2_sea_corr:+.4f}
  CO2 Emissions    vs Avg Temperature : {temp_co2_corr:+.4f}
  Forest Area %    vs Avg Temperature : {forest_temp_corr:+.4f}
  Renewable Energy vs CO2 Emissions   : {renew_co2_corr:+.4f}

  [Cell 7] Anomaly Detection
  Extreme Records (>= 1 anomaly)      : {extreme:,} / {total:,}  ({100*extreme/total:.1f}%)

  [Cell 9] Linear Regression (Predict Avg Temperature)
  RMSE                                : {rmse:.4f} degC
  MAE                                 : {mae:.4f} degC
  R-squared                           : {r2:.4f}

  [Cell 10] K-Means Clustering (K=4)
  Silhouette Score                    : {silhouette:.4f}
  Risk Labels                         : High, Med-High, Med-Low, Low Risk

  [Cell 11] Tableau Exports (7 CSV files)
  Saved to: ./tableau_exports/
''')

spark.stop()
print('SparkSession stopped. All done!')

  ANALYSIS COMPLETE — KEY FINDINGS SUMMARY
  Student: Bipin Baruwal  |  ID: 190438

  Dataset
  -------
  Records    : 1,000
  Countries  : 15
  Year Range : 2000 - 2023  (24 years)

  [Cell 5] Correlation Analysis
  CO2 Emissions    vs Sea Level Rise  : -0.0388
  CO2 Emissions    vs Avg Temperature : +0.0123
  Forest Area %    vs Avg Temperature : -0.0170
  Renewable Energy vs CO2 Emissions   : -0.0234

  [Cell 7] Anomaly Detection
  Extreme Records (>= 1 anomaly)      : 0 / 1,000  (0.0%)

  [Cell 9] Linear Regression (Predict Avg Temperature)
  RMSE                                : 8.5251 degC
  MAE                                 : 7.3221 degC
  R-squared                           : -0.0186

  [Cell 10] K-Means Clustering (K=4)
  Silhouette Score                    : 0.2165
  Risk Labels                         : High, Med-High, Med-Low, Low Risk

  [Cell 11] Tableau Exports (7 CSV files)
  Saved to: ./tableau_exports/

SparkSession stopped. All done!
